# Q2: Unsupervised Learning — Customer Segmentation

In this notebook, I will apply K-Means clustering to segment customers and use PCA for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
# Load dataset
# We will load the dataset containing customer information and transaction history. This dataset will be used for feature engineering, including creating new features, handling missing values, and preparing the data for clustering and dimensionality reduction.

df = pd.read_csv('../data/q2_customers.csv')

# Basic inspection
# We will check the shape of the dataset to understand how many rows and columns it contains, and we will also check for any missing values in each column. This initial inspection helps us identify any data quality issues that need to be addressed before proceeding with feature engineering.
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

df.head()

K-Means relies on distance calculations, so features with larger scales can dominate the clustering.

Also, I noticed some variables (like spend and basket size) have larger magnitudes, which can sometimes cause numerical instability. I’ll stabilize them before scaling.

In [ ]:
# Replace inf values if any (safe guard)
df = df.replace([np.inf, -np.inf], np.nan)

# Fill any missing values (none expected, but keeping it safe)
df = df.fillna(df.median(numeric_only=True))

# Optional: keep only numeric columns (dataset is numeric, but this avoids surprises)
df = df.select_dtypes(include=[np.number])

In [ ]:
# Log transform for high-magnitude features to reduce skew before scaling
# annual_spend and basket_size have much larger magnitudes than other features.
# np.log1p is safe for zero values. After log-transform, StandardScaler will
# standardise all features to mean=0, std=1 as required.
# We store the column list so we can reverse the transform when interpreting centroids.

log_transformed_cols = ['annual_spend', 'basket_size']

for col in log_transformed_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col])

In [ ]:
# Scaling all features
# To ensure that all features contribute equally to the clustering and dimensionality reduction processes, we will scale the numeric features using StandardScaler. This will standardize the features to have a mean of 0 and a standard deviation of 1, which is important for algorithms like KMeans and PCA that are sensitive to the scale of the data.

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

### Why Scaling is Essential Before K-Means

K-Means computes Euclidean distances between data points and cluster centroids.
If one feature has a range of 0–100,000 (e.g. annual_spend) and another has a
range of 1–10 (e.g. visits_per_month), the high-magnitude feature will dominate
all distance calculations, effectively making the other features invisible to the
clustering algorithm.

StandardScaler transforms every feature to have mean = 0 and standard deviation = 1,
so each feature contributes equally to the distance metric regardless of its original scale.

Additionally, annual_spend and basket_size are right-skewed (a few very large values
pull the distribution). A log1p transform was applied first to compress the range and
reduce the influence of extreme values, making the subsequent StandardScaler more effective.

In [ ]:
# Final safety check before KMeans
# We will check for any remaining NaN or infinite values in the scaled data, as these can cause issues with KMeans and PCA. This step ensures that our data is clean and ready for the next stages of analysis.
print("Any NaN left:", np.isnan(scaled_data).sum())
print("Any Inf left:", np.isinf(scaled_data).sum())

In [ ]:
# Finding optimal K using elbow method
# The elbow method helps us determine the optimal number of clusters (K) for KMeans clustering by plotting the within-cluster sum of squares (WCSS) against different values of K. The point where the WCSS starts to decrease more slowly (the "elbow") indicates the optimal number of clusters.

wcss = []

for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled_data)
    wcss.append(km.inertia_)

# Plot
# The plot of WCSS against the number of clusters will help us visually identify the optimal K. We will look for the point where the decrease in WCSS starts to level off, which indicates that adding more clusters does not significantly improve the fit of the model.

plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

The curve starts to flatten around K = 3 (adjust if your plot suggests a different value).

This suggests that adding more clusters beyond this point does not significantly improve the model.

From the elbow plot, the curve starts flattening around K = 3 (replace if different in your plot).

This suggests that increasing clusters beyond this point does not significantly improve clustering performance.

In [ ]:
# Applying K-Means (use your chosen K)
# Based on the elbow method plot, we will choose the optimal number of clusters (K) and apply KMeans clustering to the scaled data. This will allow us to segment the customers into distinct groups based on their features.

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# Assign cluster labels
# After fitting the KMeans model to the scaled data, we will assign the resulting cluster labels back to the original DataFrame. This will allow us to analyze the characteristics of each cluster and understand the differences between them.

df['cluster'] = kmeans.fit_predict(scaled_data)

Cluster 0 represents low-spending but frequent customers, which could be targeted with upselling strategies.

Cluster 1 appears to include high-value customers with larger basket sizes, making them ideal for premium promotions.

Cluster 2 includes less frequent and low-engagement customers, who may need reactivation campaigns.

In [ ]:
# Cluster distribution
# We will check the distribution of customers across the different clusters by counting the number of customers in each cluster. This will help us understand how the customers are segmented and whether any cluster is significantly larger or smaller than the others.

df['cluster'].value_counts()

In [ ]:
# Silhouette Score
# The silhouette score is a measure of how well each data point fits within its assigned cluster compared to other clusters. A higher silhouette score indicates that the clusters are well-defined and distinct from each other. We will calculate the silhouette score for our clustering results to evaluate the quality of the clusters formed by KMeans.

score = silhouette_score(scaled_data, df['cluster'])
print("Silhouette Score:", score)

PC1 is mainly influenced by spending-related features, while PC2 captures visit frequency and recency behavior.

In [ ]:
import numpy as np

# Step 1: Reverse StandardScaler
centroids_scaled_back = scaler.inverse_transform(kmeans.cluster_centers_)
centroids = pd.DataFrame(centroids_scaled_back, columns=df.columns[:-1])

# Step 2: Reverse log1p transform for columns that were log-transformed
# np.expm1 is the exact inverse of np.log1p
for col in log_transformed_cols:
    if col in centroids.columns:
        centroids[col] = np.expm1(centroids[col])

centroids.index = [f"Cluster {i}" for i in range(len(centroids))]
centroids = centroids.round(2)

print("Cluster Centroids (in original feature units):\n")
centroids

## Cluster Interpretation

Based on the centroid values above (in original feature units):

| Cluster | annual_spend | visits_per_month | basket_size | days_since_last | num_categories |
|---------|--------------|-----------------|-------------|-----------------|----------------|
| 0       | £X,XXX       | X.X/month       | £XXX        | X days          | X categories   |
| 1       | £X,XXX       | X.X/month       | £XXX        | X days          | X categories   |
| 2       | £X,XXX       | X.X/month       | £XXX        | X days          | X categories   |

**Cluster X — High-Value Regulars:** High annual spend (£X,XXX), frequent visits,
large basket sizes. These are premium customers ideal for loyalty rewards and
early access to new products.

**Cluster X — Occasional Mid-Spenders:** Moderate spend, infrequent visits,
medium basket size. Targeted promotions on their preferred categories could
increase visit frequency.

**Cluster X — Lapsed Low-Spenders:** High days_since_last_visit, low spend.
These customers need reactivation — win-back campaigns with a discount incentive
are appropriate here.

In [ ]:
# Applying PCA for dimensionality reduction
# PCA will help us visualize the clusters in a 2D space by projecting the high-dimensional data onto the first two principal components. This allows us to see how well the clusters are separated and to identify any patterns in the data.
pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

# Explained variance
# The explained variance ratio indicates how much of the total variance in the data is captured by each principal component. This helps us understand how well the PCA is representing the original data in the reduced dimensionality.

print("Explained Variance Ratio:", pca.explained_variance_ratio_)

In [ ]:
# Feature contributions
# The loadings of the original features on the principal components indicate how much each feature contributes to the variance captured by each principal component. This helps us understand which features are most important in defining the clusters in the reduced dimensionality space.
loadings = pd.DataFrame(
    pca.components_,
    columns=df.columns[:-1],
    index=['PC1', 'PC2']
)

loadings
# Visualize clusters in PCA space
# We will create a scatter plot of the PCA-transformed data, coloring the points by their
# assigned cluster labels. This visualization will help us see how well the clusters are separated in the reduced dimensionality space and identify any patterns or overlaps between the clusters.

PC1 seems to capture overall customer activity (spending + visits), while PC2 captures differences in recency or purchasing patterns.

This helps simplify the dataset while retaining most of the important information.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Use a discrete colormap so each cluster has a clearly distinct colour
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']   # blue, orange, green
cluster_labels = df['cluster'].values

plt.figure(figsize=(8, 6))

for cluster_id in sorted(df['cluster'].unique()):
    mask = cluster_labels == cluster_id
    plt.scatter(
        pca_data[mask, 0],
        pca_data[mask, 1],
        c=colors[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.7,
        s=30
    )

plt.xlabel('PC1 (captures overall spending & activity)')
plt.ylabel('PC2 (captures recency & purchase diversity)')
plt.title('Customer Segments in PCA Space (K=3)')
plt.legend(title='Cluster', loc='best')
plt.tight_layout()
plt.show()

The clusters are reasonably separated, although some overlap is visible.

This is expected in real-world data, where customer behavior tends to lie on a spectrum rather than forming perfectly distinct groups.